# BHP Group (ASX: BHP) — Equity Research Analysis

**A walkthrough of the full model: data → 3-statement model → DCF → comps → outputs**

This notebook is the narrative layer of the project. The actual model logic lives in `src/`; this notebook imports and runs it, then visualises the results so the methodology is clear and reproducible.

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
plt.rcParams.update({'font.family': 'Arial', 'axes.spines.top': False, 'axes.spines.right': False})

NAVY = '#1F4E78'
BLUE = '#2E75B6'
AMBER = '#F9A825'
GREEN = '#2E7D32'
GREY = '#595959'

print('Setup complete.')

---
## 1. Data Fetch

`data_fetch.py` tries `yfinance` first, falls back to the cached FY2025 snapshot if offline.

In [ ]:
import data_fetch

snapshot = data_fetch.get_financials()
print('Source:', snapshot['source'])

company = snapshot['company']
print(f"\nCompany: {company['name']} ({company['ticker']})")
print(f"Sector:  {company['sector']}")
print(f"Market Cap (AUD): A${company['market_cap_aud_m']:,.0f}m")
print(f"Share Price:       A${company['share_price_aud']:.2f}")

---
## 2. Three-Statement Model

**Concept:**  
The three statements are *linked* — changing one assumption (e.g. revenue growth) ripples through all three.

Key links:
- Net Income → Retained Earnings (Balance Sheet) and starting point of Cash Flow Statement  
- Depreciation (non-cash) → added back in Cash Flow  
- Capex → reduces Cash Flow, increases PP&E on Balance Sheet  
- Ending Cash from Cash Flow = Cash on Balance Sheet

In [ ]:
import three_statement_model as tsm

model = tsm.build_model(snapshot)

print('=== INCOME STATEMENT (USDm) ===')
display(model['income_statement'])

print('\n=== BALANCE SHEET (USDm) ===')
display(model['balance_sheet'])

print('\n=== CASH FLOW STATEMENT (USDm) ===')
display(model['cash_flow'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('BHP Group — Key Financial Trends (USDm)', fontsize=13, fontweight='bold', color=NAVY)

years = model['income_statement'].columns.tolist()
actual_mask = [y.endswith('A') for y in years]
forecast_mask = [y.endswith('E') for y in years]

rev = model['income_statement'].loc['Revenue']
ebitda = model['income_statement'].loc['EBITDA']
fcf = model['cash_flow'].loc['Free Cash Flow']

for ax, series, title in zip(axes, [rev, ebitda, fcf], ['Revenue', 'EBITDA', 'Free Cash Flow']):
    colors = [NAVY if y.endswith('A') else BLUE for y in years]
    bars = ax.bar(years, series.values / 1000, color=colors, alpha=0.85, width=0.6)
    ax.set_title(title, fontweight='bold', color=NAVY)
    ax.set_ylabel('USD billions')
    ax.axvline(x=1.5, color=GREY, linestyle='--', linewidth=0.8, alpha=0.6)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=7)

legend_handles = [mpatches.Patch(color=NAVY, label='Actual'), mpatches.Patch(color=BLUE, label='Forecast')]
fig.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../output/chart_financials.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. DCF Valuation

**Concept:**  
DCF values the company based on the cash it will generate for its owners, discounted to today's value using the WACC.

```
Enterprise Value = Σ [FCF_t / (1 + WACC)^t]  +  TV / (1 + WACC)^5
Terminal Value   = FCF_5 × (1 + g) / (WACC − g)   [Gordon Growth]
Equity Value     = EV − Net Debt
Fair Value/Share = Equity Value / Shares Outstanding
```

**WACC** = cost of equity (CAPM) × equity weight + after-tax cost of debt × debt weight

In [ ]:
import valuation as val

dcf = val.run_dcf(model, snapshot)

print(f"WACC:                      {dcf['wacc']:.2%}")
print(f"Terminal Growth Rate:       {dcf['terminal_growth_rate']:.1%}")
print(f"Sum of PV of FCFs (USDm):  {dcf['sum_pv_fcfs']:,.0f}")
print(f"PV of Terminal Value:       {dcf['pv_terminal_value']:,.0f}")
print(f"Enterprise Value:           {dcf['enterprise_value']:,.0f}")
print(f"(-) Net Debt:               {dcf['net_debt']:,.0f}")
print(f"Equity Value:               {dcf['equity_value']:,.0f}")
print(f"\nImplied Share Price (AUD):  A${dcf['implied_share_price_aud']:.2f}")
print(f"Current Share Price (AUD):  A${dcf['current_share_price_aud']:.2f}")
print(f"Upside / (Downside):        {dcf['upside_downside_pct']:+.1%}")

In [ ]:
# Waterfall chart: EV bridge
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('DCF Valuation — BHP Group', fontsize=13, fontweight='bold', color=NAVY)

# Left: EV bridge
ax = axes[0]
labels = ['PV of FCFs', 'PV of Terminal Value', 'Enterprise Value']
vals = [dcf['sum_pv_fcfs'] / 1000, dcf['pv_terminal_value'] / 1000, dcf['enterprise_value'] / 1000]
colors = [BLUE, AMBER, NAVY]
bars = ax.bar(labels, vals, color=colors, alpha=0.9, width=0.5)
ax.set_title('Enterprise Value Bridge (USDbn)', fontweight='bold', color=NAVY)
ax.set_ylabel('USD billions')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'${v:.0f}bn', ha='center', fontsize=9, fontweight='bold')
ax.tick_params(axis='x', rotation=10)

# Right: Sensitivity heatmap (WACC vs g)
ax2 = axes[1]
wacc_range = [dcf['wacc'] - 0.01, dcf['wacc'] - 0.005, dcf['wacc'], dcf['wacc'] + 0.005, dcf['wacc'] + 0.01]
g_range = [dcf['terminal_growth_rate'] - 0.005, dcf['terminal_growth_rate'], dcf['terminal_growth_rate'] + 0.005]
fcf_f = model['cash_flow'].loc['Free Cash Flow']
fys = [c for c in fcf_f.index if c.endswith('E')]

sens_matrix = []
for w in wacc_range:
    row = []
    for g in g_range:
        pv = sum(fcf_f[y] / (1 + w) ** (i + 1) for i, y in enumerate(fys))
        tv = fcf_f[fys[-1]] * (1 + g) / (w - g)
        pv_tv = tv / (1 + w) ** len(fys)
        eq = pv + pv_tv - dcf['net_debt']
        row.append((eq / company['shares_outstanding_m']) / company['aud_usd_fx'])
    sens_matrix.append(row)

import numpy as np
im = ax2.imshow(sens_matrix, cmap='RdYlGn', aspect='auto')
ax2.set_xticks(range(len(g_range)))
ax2.set_yticks(range(len(wacc_range)))
ax2.set_xticklabels([f'{g:.1%}' for g in g_range])
ax2.set_yticklabels([f'{w:.2%}' for w in wacc_range])
ax2.set_xlabel('Terminal Growth Rate', fontweight='bold')
ax2.set_ylabel('WACC', fontweight='bold')
ax2.set_title('Sensitivity: Implied Price (AUD)', fontweight='bold', color=NAVY)
for i in range(len(wacc_range)):
    for j in range(len(g_range)):
        ax2.text(j, i, f'A${sens_matrix[i][j]:.1f}', ha='center', va='center', fontsize=8, fontweight='bold')
ax2.add_patch(plt.Rectangle((0.5, 1.5), 1, 1, fill=False, edgecolor='black', lw=2))

plt.colorbar(im, ax=ax2, label='Implied Price (AUD)')
plt.tight_layout()
plt.savefig('../output/chart_dcf.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Comparable Company Analysis (Comps)

**Concept:**  
If similar companies trade at a certain EV/EBITDA or P/E multiple, we apply that same multiple to BHP's financials to estimate what the market *should* pay for BHP.

- **EV/EBITDA** — enterprise value multiple; ignores capital structure differences between companies  
- **P/E** — equity value multiple; sensitive to leverage and tax rates

In [ ]:
comps = val.run_comps(model, snapshot)

print('=== PEER GROUP ===')
display(comps['peers'])

print(f"\nPeer avg EV/EBITDA:         {comps['peer_avg_ev_ebitda']:.1f}x")
print(f"BHP current EV/EBITDA:      {comps['bhp_current_ev_ebitda']:.1f}x   (PREMIUM vs peers)")
print(f"\nPeer avg P/E:               {comps['peer_avg_pe']:.1f}x")
print(f"BHP current P/E:            {comps['bhp_current_pe']:.1f}x")

print(f"\nImplied price (EV/EBITDA avg): A${comps['implied_price_ev_avg_aud']:.2f}")
print(f"Implied price (P/E avg):       A${comps['implied_price_pe_avg_aud']:.2f}")
print(f"Current share price:           A${comps['current_share_price_aud']:.2f}")

In [ ]:
# Football field chart — valuation summary
fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle('BHP — Valuation Football Field (AUD per share)', fontsize=12, fontweight='bold', color=NAVY)

methods = [
    ('DCF', dcf['implied_share_price_aud'] * 0.85, dcf['implied_share_price_aud'] * 1.15, dcf['implied_share_price_aud']),
    ('EV/EBITDA Comps (avg)', comps['implied_price_ev_avg_aud'] * 0.90, comps['implied_price_ev_avg_aud'] * 1.10, comps['implied_price_ev_avg_aud']),
    ('P/E Comps (avg)', comps['implied_price_pe_avg_aud'] * 0.90, comps['implied_price_pe_avg_aud'] * 1.10, comps['implied_price_pe_avg_aud']),
]

colors_list = [BLUE, AMBER, GREEN]
for i, (method, low, high, mid) in enumerate(methods):
    ax.barh(i, high - low, left=low, height=0.4, color=colors_list[i], alpha=0.7)
    ax.plot(mid, i, 'o', color='black', zorder=5, markersize=7)
    ax.text(high + 0.5, i, f'A${mid:.2f}', va='center', fontsize=9, fontweight='bold')

ax.axvline(x=comps['current_share_price_aud'], color='red', linestyle='--', linewidth=1.5, label=f"Current Price: A${comps['current_share_price_aud']:.2f}")
ax.set_yticks(range(len(methods)))
ax.set_yticklabels([m[0] for m in methods])
ax.set_xlabel('Implied Share Price (AUD)')
ax.legend(fontsize=9)
ax.set_xlim(20, 75)
plt.tight_layout()
plt.savefig('../output/chart_football_field.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Generate Outputs

Run the cell below to regenerate the Excel workbook and PDF memo.

In [ ]:
from build_excel import main as build_excel
from build_memo import build_memo

build_excel()
build_memo(snapshot, model, dcf, comps, '../output/BHP_Investment_Memo.pdf')

print('Done. Check the output/ folder.')

---
## 6. Recommendation

**SELL** — based on blended DCF and comps analysis.

BHP is a high-quality business (world's lowest-cost major iron ore producer, growing copper exposure), but its current share price appears to embed optimistic assumptions that leave limited margin of safety:

- BHP trades at ~8.2x EV/EBITDA vs a peer average of ~4.8x — a meaningful premium
- DCF implies ~28% downside to current price
- Elevated capex cycle (Jansen potash) weighs on near-term free cash flow

A re-rating toward peer multiples or a softer commodity price environment could compress the share price materially.